In [1]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

df = pd.read_csv("ab_data.csv")
print("Raw rows:", len(df))

# 1. Data quality: keep only rows where the group matches the page shown, then dedupe users
ok = ((df.group == "control") & (df.landing_page == "old_page")) | \
     ((df.group == "treatment") & (df.landing_page == "new_page"))
print("Mismatched rows removed:", (~ok).sum())
df = df[ok]
dupes = df.user_id.duplicated().sum()
print("Duplicate users removed:", dupes)
df = df.drop_duplicates("user_id")
print("Clean users:", len(df))

# 2. Conversion by group
g = df.groupby("group")["converted"].agg(conversions="sum", users="count")
g["rate"] = g.conversions / g.users
print(g)

# 3. Two-proportion z-test (treatment vs control)
z, p = proportions_ztest(g.loc[["treatment", "control"], "conversions"],
                         g.loc[["treatment", "control"], "users"])
lift = g.loc["treatment", "rate"] - g.loc["control", "rate"]
print(f"Absolute lift: {lift:.4%}  z={z:.2f}  p={p:.4f}")
for grp in ["control", "treatment"]:
    lo, hi = proportion_confint(g.loc[grp, "conversions"], g.loc[grp, "users"])
    print(grp, f"95% CI: {lo:.4%} – {hi:.4%}")

Raw rows: 294478
Mismatched rows removed: 3893
Duplicate users removed: 1
Clean users: 290584
           conversions   users      rate
group                                   
control          17489  145274  0.120386
treatment        17264  145310  0.118808
Absolute lift: -0.1578%  z=-1.31  p=0.1899
control 95% CI: 11.8713% – 12.2060%
treatment 95% CI: 11.7144% – 12.0472%


In [2]:
import numpy as np
c, t = g.loc["control"], g.loc["treatment"]
se = np.sqrt(c.rate*(1-c.rate)/c.users + t.rate*(1-t.rate)/t.users)
lo, hi = lift - 1.96*se, lift + 1.96*se
print(f"95% CI of the lift: {lo:.3%} to {hi:.3%}")
print(f"Relative lift: {lift / c.rate:.2%}")

95% CI of the lift: -0.394% to 0.078%
Relative lift: -1.31%
